In [136]:
import openai
import pandas as pd
import json

In [137]:
description = "1 BLIND FLANGE, CL2500, RF, ASTM A105N, ASME B16.5, SOUR SERVICE"


In [ ]:
# Set your API key securely
openai.api_key = "OPENAI_API_KEY"

In [139]:
filepath="ICE ENHANCEMENT PROJECT.xlsx"

Load Template sheet

In [140]:
def load_template_sheets(filepath):
    pipe_template = pd.read_excel(filepath, sheet_name="Pipe_Template", header=None)
    flange_template = pd.read_excel(filepath, sheet_name="Flange_template", header=None)
    return pipe_template, flange_template

Identify product type based on description

In [ ]:
def identify_product_type(description):
    prompt = f"""
You are an expert in identifying product types and extracting structured attributes from product descriptions.
## Identify Product Type
You will be given a product description.
Identify the product type from the following list:
- Pipe
- Flange
If you cannot determine the product type, return: "UNKNOWN"

## Input:
Item Description:
{description}
    """
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "system", "content": prompt}],
            temperature=0.3,
            max_tokens=100,
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception as e:
        print(f"LLM call failed: {e}")
        return "UNKNOWN"

product_type = identify_product_type(description)
print("Product type:", product_type)

Product type: Product Type: Flange


Extract column name from product template

In [142]:
def extract_column_names(pipe_template, flange_template):
    pipe_columns = pipe_template.iloc[1].tolist()
    flange_columns = flange_template.iloc[0].tolist()
    
    pipe_data = pipe_template.iloc[2:].reset_index(drop=True)
    flange_data = flange_template.iloc[1:].reset_index(drop=True)

    pipe_dict = {col: pipe_data.iloc[:, i].dropna().tolist() for i, col in enumerate(pipe_columns)}
    flange_dict = {col: flange_data.iloc[:, i].dropna().tolist() for i, col in enumerate(flange_columns)}
    # print ("efhowehkldhr", flange_dict)

    return pipe_columns, flange_columns, pipe_dict, flange_dict

Mpping-values and Attributes(pipe and flange)

In [143]:
def map_pipe_attributes(description, pipe_columns,pipe_dict):
    product_term1 = ", ".join(f'"{term}"' for term in pipe_dict.get('PRODUCT', []))
    norm_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('NORM', []))
    construction_term1 = ", ".join(f'"{term}"' for term in pipe_dict.get('CONSTRUCTION', []))
    size1_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('SIZE1', []))
    schedule_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('SCHEDULE', []))
    grade_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('GRADE', []))
    level_class_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('LEVEL_CLASS', []))
    material_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('MATERIAL', []))
    length_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('LENGTH', []))
    ends_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('ENDS', []))
    wall_thickness_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('WALL_THICKNESS', []))
    outer_diameter_terms1= ", ".join(f'"{term}"' for term in pipe_dict.get('OUTER_DIAMETER', []))
    coating_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('COATING', []))
    dimen_stand_terms1 = ", ".join(f'"{term}"' for term in pipe_dict.get('DIMEN_STAND', []))
    prompt = f"""
You are an expert in extracting attributes for PIPE products.

Your task is to extract structured attribute values from a product description using the column headers and their definitions provided below. If any attribute value is not found, return "NA".

---
Attribute Definitions:
- PRODUCT: If the description contains any of the PIPE-related terms: (e.g.,{product_term1}), return "PIPE".
- NORM: Extract the standard or specification code (e.g., {norm_terms1}). If the standard starts with "ASTM", exclude "ASTM" and return only the code part (e.g., from "ASTM B423", extract "B423",from "ASTM A333-6", extract "A333",from"ASTM A106-B",extract "A106").
- CONSTRUCTION: The method of pipe manufacturing or forming (e.g., {construction_term1}).
- SIZE1: The nominal diameter or pipe size (e.g.,{size1_terms1}). May be given in inches or millimeters (as OD).
- SCHEDULE: Indicates the wall thickness or pressure rating (e.g., {schedule_terms1}). A higher schedule number means thicker walls. If the value starts with "SCH-", "SCH ", or "SCHEDULE ", remove the prefix and return only the actual schedule (e.g., from "SCH-XXS" or "SCHEDULE 80", return "XXS" or "80").
- GRADE: Material grade or strength classification (e.g., {grade_terms1}). It identifies material composition and mechanical strength.
- LEVEL_CLASS: The quality level or class of the pipe (e.g.,{level_class_terms1}).
- MATERIAL:Extract the base material category from the input text (e.g., {material_terms1}).Only extract if the material is explicitly mentioned in the input.
- LENGTH: Pipe length type (e.g., {length_terms1}(e.g., in inches or feet).
- ENDS: Return the short code representing the pipe end type (e.g.,{ends_terms1}, BE = Beveled End, PE = Plain End, T&C = Threaded & Coupled). Always return the short form such as "BE", "PE", or "T&C" even if the full form is given in the description.
- WALL_THICKNESS: Thickness of the pipe wall in millimeters or inches (e.g.,{wall_thickness_terms1}).
- OUTER_DIAMETER: Outer diameter of the pipe in millimeters or inches(e.g.,{outer_diameter_terms1}).
- COATING: External or internal protective coating applied (e.g., {coating_terms1}).
- DIMEN_STAND:  Identify and extract the full dimensional standard followed (e.g.,{dimen_stand_terms1}).Ensure the extracted value includes both the organization name (e.g., "ASME", "EN ISO") and the standard number if mentioned together.

---

📊 Example Contextual Data (for reference):
1. "12" SMLS PIPE API 5L GRADE B PSL2 SCH 40 BE SRL ASME B36.19M "
   → PRODUCT: PIPE, SIZE1: 12", CONSTRUCTION: SMLS, NORM: API 5L, GRADE: GRADE B, LEVEL_CLASS: PSL2, SCHEDULE: SCH 40, ENDS: BE, LENGTH: SRL,DIMEN_STAND: ASME B36.19M

2. "OD 219MM ERW PIPE ISO 3183 GR X52 PSL1, 6M LENGTH, PE"
   → PRODUCT: PIPE, SIZE1: OD 219MM, CONSTRUCTION: ERW, NORM: ISO 3183, GRADE: X52, LEVEL_CLASS: PSL1, LENGTH: 6, ENDS: PE, DIMEN_STAND: EN ISO 1127 D2/T3

---

🧾 Description: "{description}"

📋 Pipe Columns:
{pipe_columns}

Respond with a valid JSON dictionary using the given column names.
"""
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You extract PIPE product attributes from descriptions."},
                {"role": "user", "content": prompt}
            ]
        )
        result = response['choices'][0]['message']['content']
        return json.loads(result)
    except Exception as e:
        print(f"Error in PIPE mapping: {e}")
 

In [144]:
def map_flange_attributes(description, flange_columns,flange_dict):
    product_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('PRODUCT', []))
    norm_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('NORM', []))
    construction_term2 = ", ".join(f'"{term}"' for term in flange_dict.get('CONSTRUCTION', []))
    size1_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('SIZE1', []))
    schedule_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('SCHEDULE', []))
    grade_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('GRADE', []))
    size2_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('SIZE2', []))
    schedule2_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('SCHEDULE2', []))
    pressure_class_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('PRESSURE_CLASS', []))
    level_class_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('LEVEL_CLASS', []))
    material_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('MATERIAL', []))
    ends_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('ENDS', []))
    wall_thickness_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('WALL_THICKNESS', []))
    wall_thickness2_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('WALL_THICKNESS2', []))
    outer_diameter_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('OUTER_DIAMETER', []))
    outer_diameter2_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('OUTER_DIAMETER2', []))
    coating_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('COATING', []))
    dimen_stand_terms2 = ", ".join(f'"{term}"' for term in flange_dict.get('DIMEN_STAND', []))
    prompt = f"""
    You are an expert in extracting attributes for FLANGE products.

    Your task is to extract structured attribute values from a product description using the column headers and their definitions provided below. If any attribute value is not found, return "NA".
    ---

    Attribute Definitions:
    - **PRODUCT**: If the description contains any of the FLANGE-related terms: (e.g.,{product_terms2}), return "FLANGE".
    - **NORM**: Extract the standard or specification code (e.g.,{norm_terms2}). If the standard starts with "ASTM", exclude "ASTM" and return only the code part (e.g., from "ASTM B423", extract "B423",from "ASTM A516-60/65/70", extract "A516-60/65/70","ASTM A350-LF2",extract "A350").
    - **CONSTRUCTION**:Only return values like (e.g.,FORGED,SMLS,WELD) {construction_term2} if those exact terms appear in the description. If none are present, return "NA".
    - **SIZE1**: The primary nominal Flange size (NPS) or diameter of the flange in inches (e.g.,{size1_terms2}).
    - **SCHEDULE**:Refers to the wall thickness for the **primary size** of the flange. It is only relevant when explicitly mentioned with a prefix like "SCH-", "SCH ", or "SCHEDULE". If multiple schedule values are found in the description, always extract the **first occurrence** of such a value as `SCHEDULE`. For example, from "SCH-80", extract "80"; from "SCHEDULE 160", extract "160". If no such value is found, return `"NA"` (e.g., {schedule_terms2}).
    - **GRADE**: Material grade or alloy used in flange manufacturing (e.g., {grade_terms2}).
    - **SIZE2**: Secondary size (used in reducing or dual-size flanges), also in inches(e.g.,{size2_terms2}).
    - **SCHEDULE2**:Refers to the wall thickness for the **secondary size** of the flange. If multiple schedule values are found in the description, extract the **second occurrence** as `SCHEDULE2`.(e.g., {schedule2_terms2}).
    - **PRESSURE_CLASS**:refers to the pressure rating class of a flange, typically prefixed with "CLS", "CLASS", or "CL" followed by a number. Extract values only if they start with "CLS", "CLASS", or "CL" and are followed by a number. For example, from "CLS 150", extract "CLS 150"; from "CLASS 300", extract "CLASS 300". If no pressure class value is found, return "NA" (e.g., {pressure_class_terms2}).
    - **LEVEL_CLASS**: The quality level or performance class of the flange (e.g., {level_class_terms2}). Only extract known values such as `CL1`, `CL2`, `CL3`, `CL4`, `PSL1`, `PSL2`, `PSL3`, etc.if those exact terms appear in the description. If none are present, return "NA".
    - **MATERIAL**: Material abbreviation or category (e.g., {material_terms2}).
    - **ENDS**: End connection type (e.g.,{ends_terms2}, FF - Flat Face, RF - Raised Face, RTJ - Ring Type Joint, FNPT, BW, SW).If the value is not found, return "NA".
    - **WALL_THICKNESS**: Measured wall thickness (in mm) for primary size(e.g.,{wall_thickness_terms2}).
    - **WALL_THICKNESS2**: Measured wall thickness (in mm) for secondary size(e.g.,{wall_thickness2_terms2}).
    - **OUTER_DIAMETER**: Outer diameter (in mm) of primary size flange(e.g.,{outer_diameter_terms2}).
    - **OUTER_DIAMETER2**: Outer diameter (in mm) for the secondary size(e.g.,{outer_diameter2_terms2}).
    - **COATING**: Surface coating or treatment (e.g., {coating_terms2}).
    - **DIMEN_STAND**: Identify and extract the full dimensional standard followed (e.g.,{dimen_stand_terms2}).Ensure the extracted value includes the organization name (e.g., "ASME", "EN","API","DIN","AWWA") and the standard number if mentioned together.
    ---

    📘 Business Rule for Reducing Flanges:
    - If the description mentions **FLANGE REDUCING WELDNECK**, **FLANGE REDUCING SOCKETWELD**, **FLANGE REDUCING SLIP ON**, or **FLANGE REDUCING THREADED**, apply the following logic:
        - Identify the two sizes mentioned in the description.
        - Assign the **larger size** to `"Size-1"` and the **smaller size** to `"Size-2"`.


    Example:
    → **"FLANGE REDUCING WELDNECK 10\" x 6\" CLASS 300"**
    → **SIZE1**: 10", **SIZE2**: 6"

    📊 Example Contextual Data (for reference):

    1. **"3/4" BLIND FLANGE, CL1500, RF, WELD,ASTM A350-LF2 CL1, ASME B16.5, SOUR SERVICE"**
    → **PRODUCT**: FLANGE, **SIZE1**: 3/4",  **CONSTRUCTION**: WELD,**PRESSURE_CLASS**: CL1500, **ENDS**: RF, **NORM**:A350, **GRADE**: LF2, **LEVEL_CLASS**: CL1, **DIMEN_STAND**: ASME B16.5, **COATING**: SOUR SERVICE

    2. **"4" WELDING NECK FLANGE, CL1500, FORGED ,RF, SCH-XXS, ASTM A105N, ASME B16.5, SOUR SERVICE"**
    → **PRODUCT**:FLANGE, **SIZE1**: 4", **PRESSURE_CLASS**:CL1500, **CONSTRUCTION**:FORGED,**ENDS**: RF, **SCHEDULE**:XXS, **NORM**:A105N, **COATING**: SOUR SERVICE,**DIMEN_STAND**: ASME B16.5,

    3. **"4-1/16" SPECTACLE BLIND WITH 75MM TRANSITION PIECE, 10000 PSI, RTJ, 20.55 MM THK., ASTM A694 GR F60, API 6A TYPE 6B, SOUR SERVICE"**
    → **PRODUCT**: FLANGE, **SIZE1**: 4-1/16", **PRESSURE_CLASS**: 10000 PSI, **ENDS**: RTJ, **WALL_THICKNESS**: 20.55., **NORM**: A694, **GRADE**: F60, **DIMEN_STAND**: API 6A TYPE 6B, **COATING**: SOUR SERVICE
    
    ---

    🧾 Description: "{description}"

    📋 Flange Columns:
    {flange_columns}

    Respond with a valid JSON dictionary using the given column names.
    """  
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You extract FLANGE product attributes from descriptions."},
                {"role": "user", "content": prompt}
            ]
        )
        result = response['choices'][0]['message']['content']
        return json.loads(result)
    except Exception as e:
        print(f"Error in FLANGE mapping: {e}")
        return {}


In [145]:
def process_description(description, pipe_template, flange_template):
   
    # Extract column names from templates
    pipe_columns,flange_columns,pipe_dict,flange_dict=extract_column_names(pipe_template, flange_template)
    
    # Identify product type
    product_type_result = identify_product_type(description)
    
    # Check if product type contains "Pipe" or "Flange"
    if "PIPE" in product_type_result.upper():
        print(f"Processing as Pipe: {description}")
        return map_pipe_attributes(description, pipe_columns,pipe_dict)
    elif "FLANGE" in product_type_result.upper():
        print(f"Processing as Flange: {description}")
        return map_flange_attributes(description, flange_columns,flange_dict)
    else:
        print(f"Unknown product type: {product_type_result}")
        return {"PRODUCT": "UNKNOWN", "ERROR": f"Could not identify product type: {product_type_result}"}

In [ ]:
pipe_template, flange_template = load_template_sheets(filepath)
pipe_result = process_description(description, pipe_template, flange_template)
print(json.dumps(pipe_result, indent=2))

Processing as Flange: 1 BLIND FLANGE, CL2500, RF, ASTM A105N, ASME B16.5, SOUR SERVICE
{
  "PRODUCT": "FLANGE",
  "NORM": "A105N",
  "CONSTRUCTION": "NA",
  "SIZE1": "1",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "CLS 2500",
  "LEVEL_CLASS": "NA",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "SOUR SERVICE",
  "DIMEN_STAND": "ASME B16.5"
}


Attribute values validation in sheet 

In [147]:
def validate_attributes(attributes_dict, pipe_template, flange_template):
 
    # Extract column names
    pipe_columns, flange_columns,pipe_dict,flange_dict= extract_column_names(pipe_template, flange_template)
    
    # Determine product type - check for both exact match and substring
    product_value = attributes_dict.get("PRODUCT", "").strip().upper()
    
    # Choose correct template and columns based on product type
    if "PIPE" in product_value:
        template_df = pipe_template
        template_columns = pipe_columns
        row_start_index = 2  # Skip first two rows for pipe template
        print(f"Validating as PIPE product: {product_value}")
    elif "FLANGE" in product_value:
        template_df = flange_template
        template_columns = flange_columns
        row_start_index = 1  # Skip header row only for flange template
        print(f"Validating as FLANGE product: {product_value}")
    else:
        print(f"Unknown product type '{product_value}'; skipping validation.")
        return attributes_dict

    # Create a clean DataFrame from the valid rows
    template_data = template_df.iloc[row_start_index:].reset_index(drop=True)
    template_data.columns = template_columns

    validated_dict = {}

    # Validate each attribute against the template
    for col in template_columns:
        value = attributes_dict.get(col, "NA")
        
        # Skip validation for NA values
        if value != "NA" and value.strip() != "":
            # Get unique values from template column
            template_values = template_data[col].astype(str).str.strip().unique()
            
            # Check if value exists in template
            if str(value).strip() not in template_values:
                print(f"Value '{value}' for '{col}' not found in template - setting to NA")
                validated_dict[col] = "NA"
            else:
                validated_dict[col] = value
        else:
            validated_dict[col] = "NA"

    return validated_dict

In [148]:
extracted_attributes = process_description(description, pipe_template, flange_template)

# Validate the extracted attributes
validated_attributes = validate_attributes(extracted_attributes, pipe_template, flange_template)

# Print final result
print("\n✅ Final Validated Attributes:")
print(json.dumps(validated_attributes, indent=2))

Processing as Flange: 1 BLIND FLANGE, CL2500, RF, ASTM A105N, ASME B16.5, SOUR SERVICE
Validating as FLANGE product: FLANGE
Value 'SOUR SERVICE' for 'COATING' not found in template - setting to NA

✅ Final Validated Attributes:
{
  "PRODUCT": "FLANGE",
  "NORM": "A105N",
  "CONSTRUCTION": "NA",
  "SIZE1": "1",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "CLS 2500",
  "LEVEL_CLASS": "NA",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "NA",
  "DIMEN_STAND": "ASME B16.5"
}


In [151]:
def mandatory_attributes_validation_with_llm(attributes_dict):
    prompt = f"""
You are a product data validation expert.

Given the extracted attribute dictionary below, perform the following:
1. Identify if the product type is PIPE or FLANGE based on the "PRODUCT" field.
2. Validate that all mandatory fields for that product type are present and not "NA".
   - For PIPE: PRODUCT, NORM, GRADE, LEVEL_CLASS, SIZE1, SCHEDULE, CONSTRUCTION
   - For FLANGE: PRODUCT, NORM, GRADE, LEVEL_CLASS, SIZE1, SCHEDULE, SIZE2, SCHEDULE2, PRESSURE_CLASS
   - Other fields such as SCHEDULE2, SIZE2 may also be mandatory in some special cases.
3. If any mandatory field is missing or has the value "NA", respond with:
   "Invalid Description - Missing: <list of missing fields>"
4. If all required fields are valid, respond with:
   "Valid Description"

Only return the validation message as plain text.
---
Attributes: {json.dumps(attributes_dict, indent=2)}
"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            temperature=0.3,
            messages=[
                {"role": "system", "content": "You validate product attributes based on mandatory field rules."},
                {"role": "user", "content": prompt}
            ]
        )
        validation_message = response["choices"][0]["message"]["content"].strip()
        return validation_message
    except Exception as e:
        print(f"LLM Validation Error: {e}")
        return "Validation Error - Unable to check attributes"
    

    

In [152]:
# Step 2: Validate against template sheet values (removes invalid template matches)
validated_result = validate_attributes(extracted_attributes, pipe_template, flange_template)
print(json.dumps(validated_result, indent=2))

# Step 3: Validate business rules using LLM (based on validated result)
validation_status = mandatory_attributes_validation_with_llm(validated_result)
print("\n📋 LLM Validation Result:")
print(validation_status)

Validating as FLANGE product: FLANGE
Value 'SOUR SERVICE' for 'COATING' not found in template - setting to NA
{
  "PRODUCT": "FLANGE",
  "NORM": "A105N",
  "CONSTRUCTION": "NA",
  "SIZE1": "1",
  "SCHEDULE": "NA",
  "GRADE": "NA",
  "SIZE2": "NA",
  "SCHEDULE2": "NA",
  "PRESSURE_CLASS": "CLS 2500",
  "LEVEL_CLASS": "NA",
  "MATERIAL": "NA",
  "ENDS": "RF",
  "WALL_THICKNESS": "NA",
  "WALL_THICKNESS2": "NA",
  "OUTER_DIAMETER": "NA",
  "OUTER_DIAMETER2": "NA",
  "COATING": "NA",
  "DIMEN_STAND": "ASME B16.5"
}

📋 LLM Validation Result:
Invalid Description - Missing: GRADE, LEVEL_CLASS, SCHEDULE


In [93]:
# Step 1: Extract using LLM extractor
attributes = process_description(description, pipe_template, flange_template)
# print("Extracted Attributes:\n", json.dumps(attributes, indent=2))

# Step 2: Validate using LLM
validation_status = mandatory_attributes_validation_with_llm(validated_result)
# print("Validation:", validation_status)

# Step 3: Store in correct variable based on PRODUCT
product_type = validated_result.get("PRODUCT", "").strip().upper()

Pipe_variable = None
Flange_variable = None

if "PIPE" in product_type:
    Pipe_variable = validated_result
    print("Stored in Pipe_variable ✅")
elif "FLANGE" in product_type:
    Flange_variable = validated_result
    print("Stored in Flange_variable ✅")
else:
    print("Unknown Product Type - Not stored ❌")

Processing as Flange: 1 BLIND FLANGE, CL2500, RF, ASTM A105N, ASME B16.5, SOUR SERVICE
Stored in Flange_variable ✅


In [94]:
# Load the Norm sheet
Norm_template = pd.read_excel("ICE ENHANCEMENT PROJECT.xlsx", sheet_name="Norm-Std-pipe", header=None)
Norm_data = pipe_template.iloc[2:].reset_index(drop=True)
# Use row 1 (index 1) as the header
Norm_columns = Norm_template.iloc[1].tolist()

# Drop first two rows (typically header/title rows) and reset index
Norm_df = Norm_template.iloc[2:].reset_index(drop=True)
# Assign proper column names
Norm_df.columns = Norm_columns

# Strip any leading/trailing whitespace from column names
Norm_df.columns = Norm_df.columns.str.strip()
# Convert to list of dicts (for prompt context, you can limit to top 10 rows)
norm_data_preview = Norm_df.to_dict(orient="records")


In [ ]:
def NormBased_Attribute_Recovery(Pipe_variable, norm_data_preview):
    prompt = f"""
You are a PIPE product data validator.

Task:
- Given a PIPE input and a Norm table, match the "NORM" field from the input to a row in the Norm table.
- If a match is found, fill missing fields (marked as "NA") using corresponding values from the Norm table:
    - PRODUCT → Product Name
    - CONSTRUCTION → Construction
    - DIMEN_STAND → Dimensional STD
    - MATERIAL → Material
- Report clearly what fields were filled and what were already present.
- Show the updated PIPE input after recovery.
- Validate mandatory fields: PRODUCT, NORM, GRADE, LEVEL_CLASS, SIZE1, SCHEDULE, CONSTRUCTION.
    - If any mandatory fields are still "NA", return: "Invalid Description - Missing: [fields]"
    - Else, return: "Valid Description (with Norm reference)"

Input Pipe:
{json.dumps(Pipe_variable, indent=2)}

Norm Table:
{json.dumps(norm_data_preview, indent=2)}
"""

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4",
            temperature=0.1,
            messages=[
                {"role": "system", "content": "You are a smart validator for pipe product entries using norm tables."},
                {"role": "user", "content": prompt}
            ]
        )
        Norm_validation = response["choices"][0]["message"]["content"].strip()
        return Norm_validation
    except Exception as e:
        print(f"LLM Norm Validation Error: {e}")
        return "Norm Validation Error - Unable to check attributes"

In [247]:
Norm_based_result = NormBased_Attribute_Recovery(Pipe_variable, norm_data_preview)
print(" Norm-based Validation Result:")
print(Norm_based_result)

 Norm-based Validation Result:
As the input pipe is null, there is no data to validate or match with the Norm table. Please provide a valid input pipe for validation.


In [95]:
# Step 4: Conditional concatenation based on PRODUCT type
product_type = validated_result.get("PRODUCT", "").strip().upper()

if product_type == "FLANGE":
    # Concatenate after LLM validation for FLANGE
    concatenated = ' '.join(
        str(v) for v in validated_result.values() if str(v).strip().upper() != "NA"
    )
    print("\n🔩 FLANGE Concatenated Values (after LLM validation):")
    print(concatenated)

elif product_type == "PIPE":
    # Extract updated PIPE from norm-based LLM response
    norm_response = NormBased_Attribute_Recovery(validated_result, norm_data_preview)
    # print("\n📄 Norm-based Validation Result:")
    # print(norm_response)

    import re, json
    match = re.search(r'The updated PIPE input after recovery is:\s*({.*?})', norm_response, re.DOTALL)
    if match:
        updated_dict = json.loads(match.group(1))
        concatenated = ' '.join(
            str(v) for v in updated_dict.values() if str(v).strip().upper() != "NA"
        )
        print("\n🛠️ PIPE Concatenated Values (after norm-based recovery):")
        print(concatenated)
    else:
        print("❌ Could not extract updated PIPE input from norm response.")
else:
    print("❓ Unknown product type. Cannot concatenate.")


🔩 FLANGE Concatenated Values (after LLM validation):
FLANGE A105N 1 CLS 2500 RF ASME B16.5
